# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Two shipped findings audited, then my own model put under an honest split, a leakage audit, and a claim rewrite.

## 1. Two paper findings + my methodology questions

I audit two findings from the reference paper shipped with this repo (`docs/paper-examples/index.html`)
— not to debunk them, but to ask the same three questions I must survive: *where does the label come
from, what is the validation design, and does the design carry the claim?*

**Finding 1 (reference paper).** *"Pages that spiked to 3.3–7× their baseline month decline 69% of the
time (portfolio base: 42%)."* The label is a *next-window* decline; the frame is a development month;
the 42% base rate is the comparator.

**Finding 2 (reference paper).** *"The intuitive 'already dipping' watchlist declines at only 33% —
below the 42% base rate."* Same label, same frame; the point is that an intuitive volume-based list
can be an *anti*-signal.

**My questions:** (a) both hinge on a **volume** story — is volume a decline driver on my slice, or
only a gate? (b) both are quoted against a *base rate*; is the base rate held constant? (c) if I
re-derive the analogue on my snapshot, does the volume story survive?

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
# Window separation + the base rate both findings must be quoted against.
print("window separation: features = prior-30 window; label = later-30 window; overlap = none by construction")
print(f"my pool base rate (the comparator any finding must beat): {y.mean():.3f}")

# Finding-1 analogue: does a volume 'spike' (top prior-volume pages) decline more?
vol = pd.qcut(pool["impressions_prev_30d"], [0, .25, .75, 1.0], labels=["bottom 25%", "middle 50%", "top 25%"])
audit1 = pool.groupby(vol, observed=True).agg(n=("content_id", "count"),
                                              decline_rate=("declined_next_30d", "mean")).round(3)
print("\nFinding-1 analogue - 'big volume' pages vs the rest:")
print(audit1.to_string())

# Finding-2 analogue: does the 'already dipping' list (low prior volume) decline less?
print("\nFinding-2 analogue - the 'already dipping' (bottom-volume) list:")
print(f"  decline rate of the bottom-volume quarter: {audit1.loc['bottom 25%', 'decline_rate']:.3f} "
      f"vs pool base {y.mean():.3f}")
print("VERDICT: on my slice volume is a GATE, not a driver - the middle band is highest, so neither")
print("'big = risky' nor 'already small = safe' is a clean monotone rule here. Finding 1's mechanism")
print("does not transfer as a volume statement; Finding 2's anti-signal is only partly reproduced.")

window separation: features = prior-30 window; label = later-30 window; overlap = none by construction
my pool base rate (the comparator any finding must beat): 0.616

Finding-1 analogue - 'big volume' pages vs the rest:
                         n  decline_rate
impressions_prev_30d                    
bottom 25%            4514         0.615
middle 50%            8995         0.641
top 25%               4501         0.566

Finding-2 analogue - the 'already dipping' (bottom-volume) list:
  decline rate of the bottom-volume quarter: 0.615 vs pool base 0.616
VERDICT: on my slice volume is a GATE, not a driver - the middle band is highest, so neither
'big = risky' nor 'already small = safe' is a clean monotone rule here. Finding 1's mechanism
does not transfer as a volume statement; Finding 2's anti-signal is only partly reproduced.


**Methodology questions answered.** (a) Volume does not carry the mechanism on this snapshot — it
gates the pool. (b) Both findings are correctly quoted against a base rate; my audit does the same.
(c) The volume story does not cleanly reproduce, which is exactly why **my own** claim below is written
against a held-out base rate and a client-grouped split, not against an intuition.

## 2. My model under an honest split (before/after)

`before` = the flattering split (random pages), `after` = the honest split (whole clients held out).
*Show both numbers.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

def rf():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=10, max_features="sqrt",
                                  random_state=SEED, n_jobs=-1)

# BEFORE: random page split (leaks clients)
rt, rv = train_test_split(np.arange(len(y)), test_size=0.30, random_state=SEED, stratify=y)
rf_r = rf().fit(X.iloc[rt], y[rt])
s_r = rf_r.predict_proba(X.iloc[rv])[:, 1]
leaked = len(set(groups[rt]) & set(groups[rv]))

# AFTER: grouped by client
gt, gv = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(X, y, groups))
rf_g = rf().fit(X.iloc[gt], y[gt])
s_g = rf_g.predict_proba(X.iloc[gv])[:, 1]

print(f"BEFORE random split : P@50 {pk(s_r, y[rv])['50']} | ROC-AUC {roc_auc_score(y[rv], s_r):.3f} "
      f"| clients leaked {leaked}/{pd.Series(groups[rv]).nunique()}")
print(f"AFTER  grouped split: P@50 {pk(s_g, y[gv])['50']} | ROC-AUC {roc_auc_score(y[gv], s_g):.3f} "
      f"| clients leaked 0/{pd.Series(groups[gv]).nunique()}")
print(f"base rates: random {y[rv].mean():.3f} | grouped {y[gv].mean():.3f}")
print("\n=> random-split validation inflates the score; the grouped number is the one I may report.")

BEFORE random split : P@50 0.96 | ROC-AUC 0.720 | clients leaked 27/28
AFTER  grouped split: P@50 0.64 | ROC-AUC 0.582 | clients leaked 0/9
base rates: random 0.616 | grouped 0.567

=> random-split validation inflates the score; the grouped number is the one I may report.


## 3. Leakage audit

The same hunt as Week 3, on the final feature set: is every column knowable **before** the outcome
window, and what does one label-derived column do to the score?

In [4]:
print("--- knowability at the end of the prior-30 window ---")
overlapping = [c for c in NUM_FEATURES + CAT_FEATURES if c.endswith("_last_30d")]
print(f"feature columns drawn from the outcome window: {overlapping or 'none'}")
print(f"feature columns drawn from the 90-day totals : "
      f"{[c for c in NUM_FEATURES if c.endswith('_90d')] or 'none'}")

# the leak trap: add the outcome window back on the SAME grouped split
Xtrap = X.copy()
Xtrap["impressions_last_30d"] = raw.loc[pool.index, "impressions_last_30d"].values
rf_t = rf().fit(Xtrap.iloc[gt], y[gt])
s_t = rf_t.predict_proba(Xtrap.iloc[gv])[:, 1]
print(f"\nleak trap on the grouped split: + impressions_last_30d -> "
      f"P@50 {pk(s_t, y[gv])['50']} | AUC {roc_auc_score(y[gv], s_t):.3f}")

# five-fold grouped stability of the honest model
from sklearn.model_selection import GroupKFold
ps = []
for a, b in GroupKFold(5).split(X, y, groups):
    m = rf().fit(X.iloc[a], y[a])
    ps.append(float(pk(m.predict_proba(X.iloc[b])[:, 1], y[b])["50"]))
print(f"\nGroupKFold(5) P@50: mean {np.mean(ps):.3f} +/- {np.std(ps):.3f} across folds "
      f"| per-fold {[round(v,3) for v in ps]}")

--- knowability at the end of the prior-30 window ---
feature columns drawn from the outcome window: none
feature columns drawn from the 90-day totals : none



leak trap on the grouped split: + impressions_last_30d -> P@50 1.0 | AUC 0.864



GroupKFold(5) P@50: mean 0.788 +/- 0.148 across folds | per-fold [0.78, 0.8, 0.52, 0.88, 0.96]


## 4. Claim rewrite

*Take the boldest sentence and rewrite it in safe language: observed, measured, directional,
decision-support.*

In [5]:
rules_ps = []
for a, b in GroupKFold(5).split(X, y, groups):
    rules_ps.append(float(pk(rule[b], y[b])["50"]))
rewrites = [
    ("Claim (bold):", "My model predicts which pages will decline.",
     "SAFE:", "On held-out clients, the model ranks next-window decline at precision@50 comparable to a "
              "transparent rule, and no better (measured)."),
    ("Claim (bold):", "The rule beats machine learning.",
     "SAFE:", f"On this snapshot the rule's grouped-fold P@50 ({np.mean(rules_ps):.3f}) is within the "
              "fold spread of the forest's, so neither is clearly better (measured, one month, one slice)."),
    ("Claim (bold):", "Refresh these pages to recover traffic.",
     "SAFE:", "The queue is decision-support: it orders a human review; recovery needs a controlled test (not claimed)."),
]
for *prefix, safe in rewrites:
    print(" ".join(prefix).replace("SAFE:", ""))
    print("   ", safe)
    print()

Claim (bold): My model predicts which pages will decline. 
    On held-out clients, the model ranks next-window decline at precision@50 comparable to a transparent rule, and no better (measured).

Claim (bold): The rule beats machine learning. 
    On this snapshot the rule's grouped-fold P@50 (0.748) is within the fold spread of the forest's, so neither is clearly better (measured, one month, one slice).

Claim (bold): Refresh these pages to recover traffic. 
    The queue is decision-support: it orders a human review; recovery needs a controlled test (not claimed).



In [6]:
metrics = {
    "task": "ml-09",
    "data": "data/raw/content_refresh_anonymized.csv",
    "base_rate": round(float(y.mean()), 3),
    "section1_volume_analogue": audit1["decline_rate"].to_dict(),
    "section2_random_split": {"p50": pk(s_r, y[rv])["50"], "auc": round(float(roc_auc_score(y[rv], s_r)), 3),
                              "clients_leaked": int(leaked)},
    "section2_grouped_split": {"p50": pk(s_g, y[gv])["50"], "auc": round(float(roc_auc_score(y[gv], s_g)), 3),
                               "clients_leaked": 0},
    "section3_leak_trap": {"p50": pk(s_t, y[gv])["50"], "auc": round(float(roc_auc_score(y[gv], s_t)), 3)},
    "section3_groupkfold5": {"p50_mean": round(float(np.mean(ps)), 3), "p50_sd": round(float(np.std(ps)), 3)},
    "section3_rule_groupkfold5": {"p50_mean": round(float(np.mean(rules_ps)), 3),
                                  "p50_sd": round(float(np.std(rules_ps)), 3)},
}
(OUT / "w06_validation_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w06_validation_metrics.json")

receipt written: work/outputs/w06_validation_metrics.json


### Self-check

- [x] Two shipped findings audited with the same three questions I must survive
- [x] Both splits shown (before/after) with the leakage count printed
- [x] Leakage audit repeated on the final features, with a demonstrated leak trap
- [x] Boldest claims rewritten in safe language
- [x] Receipt written to `work/outputs/w06_validation_metrics.json`
- [x] Committed under `work/notebooks/`